# Extracting Text with OCR

**Lesson 6 — Module 1: Extraction**

In the last notebook, we saw that PyMuPDF extracts whatever text layer a PDF contains — clean or garbled. But 16% of our PDFs have no text layer at all, and some of the existing OCR layers have significant errors.

In this notebook, we'll run fresh OCR on page images to recover text from image-only PDFs, and compare the results against existing OCR layers.

## What you'll learn

- How to run OCR on a PDF using PyMuPDF's built-in method and the Tesseract CLI
- Why `--psm 6` and 300 DPI matter
- What re-OCR can and can't fix
- A three-tier extraction strategy for extraction pipelines

## 1. Install Dependencies

In [1]:
%pip install pymupdf -q

Note: you may need to restart the kernel to use updated packages.


> **Tesseract** must be installed separately on your system:
> - macOS: `brew install tesseract`
> - Linux: `apt install tesseract-ocr`
> - Windows: [Download installer](https://github.com/UB-Mannheim/tesseract/wiki)

## 2. Imports & Configuration

In [1]:
import pymupdf
import subprocess
import tempfile
import time
from pathlib import Path

PDF_DIR = Path("enron_pdfs")
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

SAMPLE_PDF = PDF_DIR / "E61D04918.pdf"    # moderate OCR errors (Bate:, ENRON CORE.)
EMPTY_PDF = PDF_DIR / "E0033CF3B.pdf"     # no text layer at all
SEVERE_PDF = PDF_DIR / "ECD0D46C3.pdf"    # severe OCR errors (COMP DDENTIAL, EMURCN CURD.)

print(f"Found {len(pdf_files):,} PDFs in {PDF_DIR.resolve()}")

Found 4,911 PDFs in /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/enron_pdfs


## 3. Recap

Recall that `E61D04918.pdf` has a garbled OCR text layer and `E0033CF3B.pdf` has no text layer at all. Run the next two cells to confirm.

In [2]:
doc = pymupdf.open(SAMPLE_PDF)
page = doc[0]
bad_text = page.get_text()

print("=== Existing OCR text layer ===")
print(bad_text[:1000])

=== Existing OCR text layer ===
CONFIDENTIAL
Enron Corp.
Case No. EC-26002-016038
Doc No. Es6lbo4918
Bate: O1/15/2003
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Kerri Thompson <kerri.thompson@enron,.
com>
Sent:
Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To:
Kate Symes <kate.symes@enron.com>
Subject :
natsource checkout
465810
matt
strike price 70.00
8.00 premium
465556
bob
broker has midc
SOAR DDERT DAL
Enron Gerp.
Case Ro. 20-7 002-91078
wea No. RETDC4I9le2
pater
Cle 5/7003
ERRGN TORP. - PRODUCED PURSUANT TO SEAC SUBPORNA.
SLBCECT TC PROTROLDIVE ORGAR. CORE OCSANYTIAL TREATKERT ReOLE
ES Tsis.
EKRON-S206 741254888



The OCR text layer has errors throughout. The PDF still has the original page image underneath. Can we re-OCR it to get better results?

### What about PDFs with no text layer at all?

Not all scanned PDFs have even a bad OCR layer. Some were scanned to image-only files — no text layer was ever created. PyMuPDF's `get_text()` returns nothing at all.

In [3]:
doc_empty = pymupdf.open(EMPTY_PDF)
page = doc_empty[0]
text = page.get_text()

print(f"File:       {EMPTY_PDF.name}")
print(f"Size:       {EMPTY_PDF.stat().st_size / 1024:.0f} KB")
print(f"Characters: {len(text.strip())}")
print(text[:200] if text.strip() else "(empty — no text layer)")

File:       E0033CF3B.pdf
Size:       1031 KB
Characters: 0
(empty — no text layer)


Nothing. The page image is there, but there's no text to extract.

These files need OCR too. Whether the text layer is **garbled** or **missing entirely**, the solution is the same: run fresh OCR on the page image. Let's look at two ways to do that.

## 4. Option 1 — PyMuPDF Built-in OCR

PyMuPDF wraps Tesseract through the `get_textpage_ocr()` method. This renders the page internally and passes it to Tesseract.

The key parameter is **`full=True`** — this forces fresh OCR on the page image, ignoring any existing text layer.

### Why 300 DPI?

300 DPI is the industry standard minimum for OCR. It captures enough detail to recognize individual character shapes reliably. Lower DPI (150) can work on clean digital renders, but scanned documents need the extra resolution to preserve fine features like serifs, dots, and thin strokes.

In [4]:
page = doc[0]
tp = page.get_textpage_ocr(language="eng", dpi=300, full=True)
builtin_text = page.get_text(textpage=tp)

print("=== PyMuPDF built-in OCR (300 DPI) ===")
print(builtin_text[:1000])

=== PyMuPDF built-in OCR (300 DPI) ===
CONFIDENTIAL
Enron
Corp.
Case
No.
EC-2002-01038
Doc
No.
E61D04918
Date:
01/15/2003
ENRON
CORP.
-
PRODUCED
PURSUANT
TO
FERC
SUBPOENA.
SUBJECT
TO PROTECTIVE
ORDER.
CONFIDENTIAL
TREATMENT REQUESTED.
RELEASE
IN
PART
From:
Kerri Thompson <kerri.thompson@enron.com>
Sent:
Wed,
22 Nov 2000
05:41:00
-0800
(PST)
To:
Kate
Symes <kate.symes@enron.com>
Subject:
natsource checkout
465810
matt
strike price
70.00
8.00 premium
465556
bob
broker
has
mid
c
CONFIDENTIAL
Enron
Corp.
Case
No.
EC-2002-01038
Doc
No.
E61D04918
Date:
01/15/2003
ENRON
CORP.
-
PRODUCED
PURSUANT
TO
FERC
SUBPOENA.
SUBJECT
TO PROTECTIVE
ORDER.
CONFIDENTIAL
TREATMENT REQUESTED.
ENRON-524674254886



The built-in OCR fragments many words onto separate lines — `From:` and the sender name appear on separate lines instead of one. This is because `get_textpage_ocr()` uses Tesseract's default page segmentation mode (PSM 3), which analyzes each text region independently.

PyMuPDF also offers a lower-level OCR path via `Pixmap.pdfocr_tobytes()` (see the [PyMuPDF OCR docs](https://pymupdf.readthedocs.io/en/latest/how-to-ocr.html)). The Tesseract CLI gives the most control — let's try that next.

## 5. Option 2 — Render + Tesseract CLI

For more control, you can render the page to an image and call Tesseract directly. This lets you tune:

- **DPI** — resolution of the rendered image
- **PSM** (Page Segmentation Mode) — how Tesseract analyzes the page layout
- **Language** — which trained model to use

We'll use `--psm 6` (assume a single uniform block of text), which works well for these single-column email PDFs.

In [5]:
page = doc[0]
pix = page.get_pixmap(dpi=300)

with tempfile.NamedTemporaryFile(suffix=".png") as f:
    pix.save(f.name)
    result = subprocess.run(
        ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
        capture_output=True, text=True,
    )
    cli_text = result.stdout

doc.close()

print("=== Tesseract CLI (300 DPI) ===")
print(cli_text[:1000])

=== Tesseract CLI (300 DPI) ===
CONFIDENTIAL

Enron Corp.

Case No. EC-2002-01038

Doc No. E61D04918

Date: 01/15/2003

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART

From: Kerri Thompson <kerri.thompson@enron.com>
Sent: Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To: Kate Symes <kate.symes@enron.com>
Subject: natsource checkout

465810

matt

strike price 70.00

8.00 premium

465556

bob

broker has mid c

CONFIDENTIAL

Enron Corp.

Case No. EC-2002-01038

Doc No. E61D04918

Date: 01/15/2003

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
ENRON-524674254886



The CLI produces similar character recognition to the built-in method (both use Tesseract on the same image), but `--psm 6` keeps lines intact — `From:` and the sender name stay on one line instead of fragmenting.

The CLI gives you explicit control over page segmentation, which is useful when experimenting with difficult scans. For bulk extraction on clean images, the built-in method is simpler and faster — no subprocess overhead, no temp files.


## 6. Side-by-side comparison

The existing text layer on `E61D04918.pdf` has errors. Both re-OCR methods run on the same underlying page image. Compare the three outputs — does re-OCR on a clean image recover what the old OCR got wrong?

In [6]:
print("=" * 60)
print("get_text() — existing bad text layer")
print("=" * 60)
print(bad_text[:500])

print(f"\n{'=' * 60}")
print("get_textpage_ocr() — PyMuPDF built-in OCR")
print("=" * 60)
print(builtin_text[:500])

print(f"\n{'=' * 60}")
print("Tesseract CLI — render + OCR")
print("=" * 60)
print(cli_text[:500])

get_text() — existing bad text layer
CONFIDENTIAL
Enron Corp.
Case No. EC-26002-016038
Doc No. Es6lbo4918
Bate: O1/15/2003
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Kerri Thompson <kerri.thompson@enron,.
com>
Sent:
Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To:
Kate Symes <kate.symes@enron.com>
Subject :
natsource checkout
465810
matt
strike price 70.00
8.00 premium
465556
bob
broker has midc
SOAR DDERT DAL
Enron Gerp.
Case Ro. 20-7 002-91078
we

get_textpage_ocr() — PyMuPDF built-in OCR
CONFIDENTIAL
Enron
Corp.
Case
No.
EC-2002-01038
Doc
No.
E61D04918
Date:
01/15/2003
ENRON
CORP.
-
PRODUCED
PURSUANT
TO
FERC
SUBPOENA.
SUBJECT
TO PROTECTIVE
ORDER.
CONFIDENTIAL
TREATMENT REQUESTED.
RELEASE
IN
PART
From:
Kerri Thompson <kerri.thompson@enron.com>
Sent:
Wed,
22 Nov 2000
05:41:00
-0800
(PST)
To:
Kate
Symes <kate.symes@enron.com>
Subject:
natsource checkout
465810
matt
strike price
70.00
8.00 premium
46555

### What changed?

The existing text layer has errors (`Bate:`, `ENRON CORE.`, `EROTECTIVE`). But re-OCR on the same page produces dramatically better results: `Date:`, `ENRON CORP.`, `PROTECTIVE`.

The underlying page image is clean. The garbled text layer came from an older or lower-quality OCR pass. Modern Tesseract, running on the clean image, corrects the errors.

This is a common situation in real document dumps: the embedded text layer is noisy, but the page image is fine. Re-OCR recovers what the original OCR missed.

### A note on image preprocessing

OCR guides often recommend **binarization** (converting greyscale to pure black and white) as a preprocessing step before Tesseract. This is useful for greyscale scans where grey background noise confuses character recognition.

Our scans happen to be B&W already -- the scanner binarized them at capture time. If your dataset contains greyscale or colour scans, binarization (Otsu's method is the standard) is a worthwhile first step before running Tesseract.

### What about a really bad scan?

`E61D04918.pdf` had moderate errors — character swaps in header fields. Some scans are far worse. `ECD0D46C3.pdf` was scanned at lower resolution, producing severely garbled text. Let's see if re-OCR helps:

In [7]:
doc_heavy = pymupdf.open(SEVERE_PDF)
page_heavy = doc_heavy[0]

# Existing text layer
heavy_bad = page_heavy.get_text()
print("=== get_text() — existing text layer ===")
print(heavy_bad[:500])

# Tesseract CLI re-OCR
pix = page_heavy.get_pixmap(dpi=300)
with tempfile.NamedTemporaryFile(suffix=".png") as f:
    pix.save(f.name)
    result = subprocess.run(
        ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
        capture_output=True, text=True,
    )
    heavy_ocr = result.stdout

doc_heavy.close()

print(f"\n{'=' * 50}")
print("=== Tesseract CLI re-OCR ===")
print(heavy_ocr[:1000])

=== get_text() — existing text layer ===
COMP DDENTIAL
Engen UlErR.
Voge Marl
rad ad FDA
Son Wo. BV OSL tl
eethert Glo bosdiv 4
EMURCN CURD. - CRUWICED LURSUANT To PERS svBRCEMA.
SUR
IR OT Ue PRSTEDTOVE SRUORPR. CCH OGREN TOAD TREATM PEPE
REQUESTED
.
RELEASE IN FULL
From:
Ward, Kim <kiwv warctenron.com>
Sent:
Tnu, 4 Get 2OOL Bfibes:3s -O/O0 (eT)
To:
Ward, Ear S (Heuster) «<.wardglbenren.
cars
Subject :
iv;
f*Rarcho oa Puorta's Yoga Progran Applaudead*
Arctber cre for you ........--e
eee ee
Fogaras, Kim
-----Original Message-----
Frem:


=== Tesseract CLI re-OCR ===
CONFIDENTIAL

Enron Corp.

Case No. EC-2002-01038

Doc No. ECDOD46C3

Date: 01/15/2003

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.

RELEASE IN FULL

From: Ward, Kim <kim.ward@enron.com>

Sent: Thu, 4 Oct 2001 23:58:38 -0700 (PDT)

To: Ward, Kim S (Houston) <.ward@enron.com>

Subject: FW: **Rancho La Puerta's Yoga Program Applauded**

Another one for you .

The existing text layer has `COMP DDENTIAL` for `CONFIDENTIAL`, `Engen UlErR.` for `Enron Corp.`, `kiwv warctenron.com` for `kim.ward@enron.com`. The text layer is severely garbled.

But re-OCR on the clean underlying image recovers the content: `CONFIDENTIAL`, `Enron Corp.`, `kim.ward@enron.com`. Even on the worst file in the dataset, the page image is clean enough for modern Tesseract to produce usable text.

The quality ceiling is the image, not the existing text layer. If the image is clean, re-OCR works regardless of how bad the old text layer is.

### OCR on empty PDFs too

Earlier we found a PDF with no text layer at all. Does OCR work on that too?

In [8]:
page = doc_empty[0]
pix = page.get_pixmap(dpi=300)

with tempfile.NamedTemporaryFile(suffix=".png") as f:
    pix.save(f.name)
    result = subprocess.run(
        ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
        capture_output=True, text=True,
    )

doc_empty.close()

print(f"=== OCR on {EMPTY_PDF.name} (was empty) ===")
print(result.stdout[:1000])

=== OCR on E0033CF3B.pdf (was empty) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0O033CF3B
Date: 01/15/2003
ENRON CORP. — PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From: Schedule Crawler <pete.davis@enron.com>
Sent: Mon, 9 Apr 2001 01:41:00 -0700 (PDT)
To: pete.davis@enron.com
Ce: bert.meyers@enron.com, bill.williams.III@enron.com,
Craig.Dean@enron.com, dporter3@enron.com, Eric.Linder@enron.com,
Geir.Solberg@enron.com, jbryson@enron.com, leaf.harasin@enron.com,
monika.causholli@enron.com, mark.guzman@enron.com,
pete.davis@enron.com, ryan.slinger@enron.com
Subject: Start Date: 4/9/01; HourAhead hour: 9; <CODESITE>
Start Date: 4/9/01; HourAhead hour: 9; No ancillary schedules awarded.
Variances detected.
Variances detected in SC Trades schedule.
LOG MESSAGES:
PARSING FILE -->> O:\Portland\WestDesk\California Scheduling\ISO Final
Schedules\2001040909.txt
---- Energy Import/Export Schedule ----
*** Fi

OCR recovers text from image-only PDFs where `get_text()` returned nothing. This is where OCR adds the most value in our pipeline -- the 795 files that had no text layer at all now have usable content.

## 7. A tiered strategy

Not every PDF needs OCR. In our dataset, 84% already have usable text layers (digital + scanned with OCR). Only the 16% that are image-only truly need fresh OCR.

The strategy: use the existing text layer where it exists, and only fall back to OCR for pages with no text at all.

In [9]:
def extract_text(pdf_path):
    """Extract text from a PDF using a per-page tiered strategy.

    For each page:
    - If get_text() returns content, use it (fast)
    - If the page is empty, fall back to built-in OCR at 300 DPI (slow)

    Returns:
        tuple: (text, page_count, ocr_pages)
    """
    doc = pymupdf.open(pdf_path)
    page_count = len(doc)
    pages = []
    ocr_pages = 0

    for page in doc:
        text = page.get_text().strip()
        if text:
            pages.append(text)
        else:
            tp = page.get_textpage_ocr(dpi=300, language="eng")
            text = page.get_text(textpage=tp).strip()
            pages.append(text)
            ocr_pages += 1

    doc.close()
    return "\n\n".join(pages), page_count, ocr_pages

In [10]:
for pdf_path in [pdf_files[0], SAMPLE_PDF, EMPTY_PDF]:
    text, pages, ocr_pages = extract_text(pdf_path)
    print(f"{pdf_path.name} — {pages} page(s), {ocr_pages} OCR'd, {len(text):,} chars")
    print(text[:300])
    print()

E0000813E.pdf — 1 page(s), 0 OCR'd, 944 chars
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000813E
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Rob Bradley <rob.bradley@enron.com>
Sent:
Tue, 21 Nov 2000 08:34:00 -0800 (PST)
To

E61D04918.pdf — 1 page(s), 0 OCR'd, 676 chars
CONFIDENTIAL
Enron Corp.
Case No. EC-26002-016038
Doc No. Es6lbo4918
Bate: O1/15/2003
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Kerri Thompson <kerri.thompson@enron,.
com>
Sent:
Wed, 22 Nov 2000 05:41:00 -08

E0033CF3B.pdf — 1 page(s), 1 OCR'd, 1,599 chars
CONFIDENTIAL
Enron
Corp.
Case
No.
EC-2002-01038
Doc
No.
E0O033CF3B
Date:
01/15/2003
ENRON
CORP.
-
PRODUCED
PURSUANT
TO
FERC
SUBPOENA.
SUBJECT
TO PROTECTIVE
ORDER.
CONFIDENTIAL
TREATMENT REQUESTED.
RELEASE
IN
PART
From:
Schedule Crawler <pete.davis@enron.co

## 8. Speed comparison

How fast is each method on its own? The cell below benchmarks all three on 50 scanned PDFs — files that have page images, so OCR has something to work with.

This is a per-method comparison, not the tiered strategy. In practice, `get_text()` handles 84% of files instantly and only the 16% image-only pages trigger OCR.

In [12]:
# Use PDFs that have page images (scanned files) for a fair OCR benchmark
scanned_sample = []
for p in pdf_files:
    doc = pymupdf.open(p)
    if doc[0].get_images():
        scanned_sample.append(p)
    doc.close()
    if len(scanned_sample) >= 50:
        break

# Method 1: Plain get_text()
start = time.perf_counter()
for pdf_path in scanned_sample:
    doc = pymupdf.open(pdf_path)
    _ = "\n\n".join(page.get_text() for page in doc)
    doc.close()
elapsed_text = time.perf_counter() - start

# Method 2: PyMuPDF built-in OCR
start = time.perf_counter()
for pdf_path in scanned_sample:
    doc = pymupdf.open(pdf_path)
    for page in doc:
        tp = page.get_textpage_ocr(language="eng", dpi=300, full=True)
        _ = page.get_text(textpage=tp)
    doc.close()
elapsed_builtin = time.perf_counter() - start

# Method 3: Tesseract CLI
start = time.perf_counter()
for pdf_path in scanned_sample:
    doc = pymupdf.open(pdf_path)
    for page in doc:
        pix = page.get_pixmap(dpi=300)
        with tempfile.NamedTemporaryFile(suffix=".png") as f:
            pix.save(f.name)
            subprocess.run(
                ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
                capture_output=True, text=True,
            )
    doc.close()
elapsed_cli = time.perf_counter() - start

n = len(scanned_sample)
print(f"{'Method':<35} {'Time':>8} {'PDFs/sec':>10}")
print("-" * 55)
print(f"{'get_text() (existing layer)':<35} {elapsed_text:>7.2f}s {n/elapsed_text:>9.1f}")
print(f"{'PyMuPDF built-in OCR':<35} {elapsed_builtin:>7.2f}s {n/elapsed_builtin:>9.1f}")
print(f"{'Tesseract CLI':<35} {elapsed_cli:>7.2f}s {n/elapsed_cli:>9.1f}")

Method                                  Time   PDFs/sec
-------------------------------------------------------
get_text() (existing layer)            0.11s     442.8
PyMuPDF built-in OCR                  45.87s       1.1
Tesseract CLI                         67.16s       0.7


### The tradeoff

Plain text extraction is roughly **250-300x faster** than OCR. But for the 795 image-only files, there is no text layer to extract -- OCR is the only option.

The tiered strategy keeps things fast: use `get_text()` for the 84% of files that have text, and reserve OCR for the 16% that don't.

| Tier | Files | Estimated time |
|---|---|---|
| Text layer (digital + scanned) | 4,116 (84%) | ~15 seconds |
| OCR (image-only) | ~800 (16%) | ~15-30 minutes |

For larger corpora, you might parallelise the OCR step or consider a vision LLM for the hardest cases.

## 9. Alternative: EasyOCR

Tesseract requires a system install (`brew install tesseract`), which can be a pain across different environments. **EasyOCR** is a Python-only alternative — pure `pip install`, no system dependencies.

It uses deep learning (CRAFT for text detection + CRNN for recognition) rather than Tesseract's classical approach. Let's try it on the same PDF.

> **Memory note:** EasyOCR uses PyTorch under the hood. At 300 DPI a letter-size page renders to ~2550×3300 pixels, which can cause an out-of-memory kernel crash on memory-constrained environments like GitHub Codespaces (8 GB RAM). The cell below caps the image at 1800 pixels on its longest side before passing it to EasyOCR. On machines with more RAM or a GPU, you can safely remove the resize step and pass the full-resolution image.

In [13]:
import easyocr
from PIL import Image

reader = easyocr.Reader(["en"], gpu=False)

doc = pymupdf.open(SAMPLE_PDF)
page = doc[0]
pix = page.get_pixmap(dpi=300)
img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

# Cap image size to avoid OOM on memory-constrained environments (e.g. Codespaces).
# On machines with more RAM or a GPU, you can remove this resize step.
max_dim = 1800
if max(img.size) > max_dim:
    img.thumbnail((max_dim, max_dim), Image.LANCZOS)

with tempfile.NamedTemporaryFile(suffix=".png") as f:
    img.save(f.name)
    results = reader.readtext(f.name, detail=0, paragraph=True)
    easy_text = "\n".join(results)

doc.close()

print("=== EasyOCR (300 DPI) ===")
print(easy_text[:1000])

### EasyOCR vs Tesseract

Compare the EasyOCR output above with the Tesseract CLI results from earlier. On these scanned emails:

- **Tesseract** handles structured headers well but can misread individual characters
- **EasyOCR** reads body text more naturally (fewer fragmented lines), but struggles more with structured fields and small text
- **EasyOCR is ~5x slower** than Tesseract on CPU

**When to choose EasyOCR:**
- You can't install Tesseract on your system (e.g. restricted environments)
- You have a GPU available (EasyOCR is significantly faster with CUDA)
- Your documents are body-text-heavy rather than structured headers

For our pipeline we'll stick with Tesseract -- structured fields (From, To, Subject, Date) are what the parser needs most.

### Other OCR options

Tesseract and EasyOCR are far from the only options. Depending on your use case, you might also consider:

- **[docTR](https://github.com/mindee/doctr)** — deep learning OCR from Mindee, good accuracy, supports PyTorch and TensorFlow
- **[PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR)** — multilingual OCR from Baidu, strong on non-Latin scripts
- **[Surya](https://github.com/VikParuchuri/surya)** — newer transformer-based OCR with layout detection
- **Cloud APIs** — Google Cloud Vision, AWS Textract, Azure AI Document Intelligence offer high accuracy but at per-page cost
- **Vision LLMs** — GPT-4o, Claude, etc. can read scanned documents directly, with the best understanding of context but the highest cost and lowest throughput

## Summary

- **OCR recovers text** from image-only PDFs that have no text layer at all -- this is where it adds the most value
- **Re-OCR on clean images** produces dramatically better results when the existing text layer is garbled -- the quality ceiling is the image, not the old text layer
- **Most scanned files** (1,955) already have usable OCR text layers
- **Tesseract CLI** with `--psm 6` gives you control over page segmentation -- useful for experimentation on difficult scans
- **PyMuPDF built-in OCR** is simpler and faster for bulk extraction -- no subprocess overhead, no temp files
- **300 DPI** is the industry standard for OCR on scanned documents
- **OCR is ~250-300x slower** than plain text extraction -- use it only where needed
- **A three-tier strategy** (digital text -> existing OCR -> fresh OCR on image-only) balances speed and coverage
- **EasyOCR** is a Python-only alternative -- no system install, better with a GPU, but slower on CPU

**Next:** Before we run the full pipeline, we'll look at combined extraction packages that handle text reading, OCR, and layout analysis in a single tool.
